# 🔬 Cell Tracking — A Stronger Baseline (Detection · Linking · **Divisions**)

The official getting-started notebook does *nearest-neighbour* tracking: aggressive `÷4` downsampling in **all** axes, a global percentile threshold, connected-components, and one-to-one Hungarian linking. It works, but it leaves a lot of score on the table.

This notebook keeps the same lightweight, **CPU-only, no-internet** spirit but fixes the three things that actually move the metric:

| | Official baseline | This notebook | Why it matters |
|---|---|---|---|
| **Z resolution** | downsampled ÷4 (±6.5 µm error) | **full Z**, only XY ÷4 | Matching tolerance is **7 µm**. Z voxels are 1.625 µm — downsampling Z alone can push a detection *out of matching range*. |
| **Detection** | global threshold + connected components | **anisotropy-aware local-maxima** on an ~isotropic grid | Connected components *merge* touching cells (under-segmentation). Peak detection separates them and directly controls density. |
| **Divisions** | ❌ none (Division-Jaccard ≈ 0) | ✅ parent → **two daughters** | The score = Edge-Jaccard **+** Division-Jaccard. Adding *any* sensible division detection is nearly free score. |
| **Linking** | Hungarian, physical distance | Hungarian + a **division pass** | One-to-one assignment structurally *cannot* represent a division. |

> **The single most important insight:** the evaluation matches predicted cells to ground-truth cells using **physically scaled** distance (`z=1.625, y=x=0.40625 µm/voxel`, cap **7 µm**). The data is **anisotropic** — Z is ~4× coarser than XY. Treat the two differently and everything downstream gets easier.


In [1]:
import os, json, glob, time
from collections import Counter

import numpy as np
import pandas as pd
import blosc2
from scipy.ndimage import gaussian_filter
from scipy.optimize import linear_sum_assignment
from skimage.feature import peak_local_max
from skimage.filters import threshold_otsu

# ----------------------------- CONFIG ---------------------------------
SCALE = np.array([1.625, 0.40625, 0.40625])  # Z, Y, X  µm/voxel (anisotropic!)

# --- detection ---
XY_DS         = 4      # downsample XY by 4 -> grid is ~isotropic (1.625 µm/voxel). Z is kept FULL.
SMOOTH_SIGMA  = 1.0    # gaussian smoothing on the isotropic grid (denoise)
MIN_PEAK_DIST = 3      # min separation between cell centres, in isotropic voxels (~4.9 µm). Lower => more cells.
THRESH_REL    = 0.30   # a peak must rise at least this fraction from background toward the brightest voxel

# --- linking ---
MAX_LINK_DIST = 12.0   # µm, gate for frame-to-frame links (how far a cell can move per frame)

# --- divisions ---  (set DETECT_DIVISIONS=False to A/B test the effect on the LB)
DETECT_DIVISIONS = True
DIV_PARENT_DIST  = 12.0  # µm, max distance parent -> second daughter
DIV_SISTER_DIST  = 7.0   # µm, two daughters of a real division start very close together
# ----------------------------------------------------------------------

# Robustly locate the test directory (the official mount, with a fallback search).
CANDIDATES = [
    '/kaggle/input/competitions/biohub-cell-tracking-during-development/test',
    '/kaggle/input/biohub-cell-tracking-during-development/test',
]
TEST_DIR = next((p for p in CANDIDATES if os.path.isdir(p)), None)
if TEST_DIR is None:
    hits = glob.glob('/kaggle/input/**/test', recursive=True)
    TEST_DIR = next((h for h in hits if glob.glob(os.path.join(h, '*.zarr'))), hits[0] if hits else CANDIDATES[0])
print('TEST_DIR =', TEST_DIR)

TEST_DIR = /kaggle/input/competitions/biohub-cell-tracking-during-development/test


## 1 · Robust volume loading

Each `.zarr` holds one array at `0/` with shape `(T, Z, Y, X)`, chunked **one timepoint per chunk** and blosc/zstd-compressed. The chunk for timepoint `t` lives at `0/c/{t}/0/0/0`. We read the metadata from `0/zarr.json`, then decompress chunks directly with `blosc2` (fast, dependency-light, and exactly what the official baseline relies on).

In [2]:
def list_datasets(test_dir):
    return sorted(d[:-5] for d in os.listdir(test_dir) if d.endswith('.zarr'))

def read_meta(zarr_path):
    meta = json.load(open(os.path.join(zarr_path, '0', 'zarr.json')))
    shape = tuple(meta['shape'])            # (T, Z, Y, X)
    dtype = np.dtype(meta['data_type'])     # e.g. uint16
    return shape, dtype

def load_volume(zarr_path, t, shape, dtype):
    """Decompress one timepoint -> (Z, Y, X) array."""
    chunk = os.path.join(zarr_path, '0', 'c', str(t), '0', '0', '0')
    raw = blosc2.decompress(open(chunk, 'rb').read())
    return np.frombuffer(raw, dtype=dtype).reshape(shape[1:])

datasets = list_datasets(TEST_DIR)
print(f'{len(datasets)} test datasets:', datasets[:6], '...' if len(datasets) > 6 else '')

4 test datasets: ['44b6_0113de3b', '44b6_0b24845f', '6bba_05b6850b', '6bba_05db0fb1'] 


## 2 · Detection — anisotropy-aware local maxima

Fluorescent cells are bright blobs with a clear intensity peak, so we detect **one local maximum per cell** instead of thresholding + connected components (which fuses neighbours).

The trick to handle anisotropy cheaply: **block-average XY by 4** (Z untouched). Since `XY_DS × 0.40625 ≈ 1.625 = Z voxel size`, the resulting grid is *physically isotropic*. On an isotropic grid a single scalar `min_distance` correctly enforces a real-world minimum spacing between cell centres — no awkward anisotropic footprints, and it runs in milliseconds per frame.

- **Full Z** keeps centroids inside the 7 µm matching window (the baseline's ÷4 Z does not).
- `MIN_PEAK_DIST` directly sets cell density → an easy knob to fight the **over-prediction penalty**.
- Block-mean (not striding) also denoises, improving peak stability.

In [3]:
def block_mean_xy(vol, f):
    """Average-pool XY by factor f, keep Z. -> ~isotropic physical grid."""
    Z, Y, X = vol.shape
    Y2, X2 = (Y // f) * f, (X // f) * f
    v = vol[:, :Y2, :X2].astype(np.float32)
    return v.reshape(Z, Y2 // f, f, X2 // f, f).mean(axis=(2, 4))

def detect(vol):
    """Return detected cell centroids as (N, 3) array of (z, y, x) in ORIGINAL voxel coords."""
    ds = block_mean_xy(vol, XY_DS)
    sm = gaussian_filter(ds, sigma=SMOOTH_SIGMA)

    # foreground threshold: Otsu, but never below a relative rise from background
    try:
        otsu = threshold_otsu(sm)
    except Exception:
        otsu = np.percentile(sm, 95)
    bg = float(np.median(sm))
    abs_th = max(otsu, bg + THRESH_REL * (float(sm.max()) - bg))

    peaks = peak_local_max(sm, min_distance=MIN_PEAK_DIST,
                           threshold_abs=abs_th, exclude_border=False)
    if peaks.size == 0:
        return np.empty((0, 3))

    out = peaks.astype(np.float64)
    # map XY back to original resolution (centre of the averaged block); Z is unchanged
    out[:, 1] = out[:, 1] * XY_DS + (XY_DS - 1) / 2.0
    out[:, 2] = out[:, 2] * XY_DS + (XY_DS - 1) / 2.0
    return out

## 3 · Linking + division detection

Frame-to-frame we solve an optimal one-to-one assignment (Hungarian) on **physically scaled** centroid distances, gated by `MAX_LINK_DIST`. That handles ordinary motion and naturally allows births/deaths (unmatched cells).

Then a dedicated **division pass** — the part the official baseline lacks entirely. A dividing cell produces *two* daughters at the next frame; one-to-one assignment can only claim one of them. So for every **unmatched** next-frame detection we look for a nearby parent that already has exactly one child, and attach it as the **second daughter** — but only when the two daughters are close to each other (real sisters separate gradually), which keeps false divisions down.

A node with ≥2 outgoing edges *is* a division under the metric, so this directly feeds Division-Jaccard. The gates (`DIV_PARENT_DIST`, `DIV_SISTER_DIST`) trade recall vs. precision — tune them on the leaderboard.

In [4]:
def link_with_divisions(prev_ids, prev_xyz, curr_ids, curr_xyz):
    """Return list of (source_id, target_id) edges. A parent may get up to two daughters."""
    edges = []
    if len(prev_ids) == 0 or len(curr_ids) == 0:
        return edges

    P = prev_xyz * SCALE
    C = curr_xyz * SCALE
    D = np.sqrt(((P[:, None] - C[None, :]) ** 2).sum(axis=2))   # physical distances (nP, nC)

    BIG = 1e6
    cost = np.where(D <= MAX_LINK_DIST, D, BIG)
    ri, ci = linear_sum_assignment(cost)

    parent_children = {}        # prev_idx -> [curr_idx, ...]
    matched_curr = set()
    for r, c in zip(ri, ci):
        if cost[r, c] < BIG:
            edges.append((prev_ids[r], curr_ids[c]))
            parent_children.setdefault(r, []).append(c)
            matched_curr.add(c)

    if DETECT_DIVISIONS:
        for c in range(len(curr_ids)):
            if c in matched_curr:
                continue
            best_p, best_d = None, np.inf
            for p in range(len(prev_ids)):
                if D[p, c] > DIV_PARENT_DIST or len(parent_children.get(p, [])) != 1:
                    continue
                sister = parent_children[p][0]
                sis_d = float(np.sqrt(((C[c] - C[sister]) ** 2).sum()))
                if sis_d <= DIV_SISTER_DIST and D[p, c] < best_d:
                    best_p, best_d = p, D[p, c]
            if best_p is not None:
                edges.append((prev_ids[best_p], curr_ids[c]))
                parent_children[best_p].append(c)
                matched_curr.add(c)
    return edges

## 4 · Build the submission

We stream through every dataset and timepoint, emitting `node` rows for detections and `edge` rows for links, exactly in the required format. Node IDs are unique **within each dataset** (re-started per dataset, which is all the metric needs).

In [5]:
def process_dataset(name):
    zarr_path = os.path.join(TEST_DIR, name + '.zarr')
    shape, dtype = read_meta(zarr_path)
    n_t = shape[0]

    rows, nid, n_div = [], 1, 0
    prev_ids, prev_xyz = [], np.empty((0, 3))

    for t in range(n_t):
        vol = load_volume(zarr_path, t, shape, dtype)
        cents = detect(vol)

        curr_ids = list(range(nid, nid + len(cents)))
        nid += len(cents)
        for k, (z, y, x) in enumerate(cents):
            rows.append({'dataset': name, 'row_type': 'node', 'node_id': curr_ids[k],
                         't': t, 'z': int(round(z)), 'y': int(round(y)), 'x': int(round(x)),
                         'source_id': -1, 'target_id': -1})

        edges = link_with_divisions(prev_ids, prev_xyz, curr_ids, cents)
        for s, tgt in edges:
            rows.append({'dataset': name, 'row_type': 'edge', 'node_id': -1, 't': -1,
                         'z': -1, 'y': -1, 'x': -1, 'source_id': s, 'target_id': tgt})

        src_counts = Counter(s for s, _ in edges)
        n_div += sum(1 for cnt in src_counts.values() if cnt >= 2)
        prev_ids, prev_xyz = curr_ids, cents

    return rows, nid - 1, n_div


all_rows = []
t0 = time.time()
for i, name in enumerate(datasets, 1):
    rows, n_nodes, n_div = process_dataset(name)
    all_rows.extend(rows)
    print(f'[{i}/{len(datasets)}] {name}: {n_nodes} nodes, {n_div} divisions  '
          f'({time.time()-t0:.0f}s elapsed)')
print(f'Total time: {time.time()-t0:.0f}s')

[1/4] 44b6_0113de3b: 14549 nodes, 18 divisions  (13s elapsed)
[2/4] 44b6_0b24845f: 8618 nodes, 20 divisions  (26s elapsed)
[3/4] 6bba_05b6850b: 3734 nodes, 1 divisions  (36s elapsed)
[4/4] 6bba_05db0fb1: 14424 nodes, 33 divisions  (49s elapsed)
Total time: 49s


In [6]:
submission = pd.DataFrame(all_rows)
submission.index.name = 'id'
submission.to_csv('submission.csv')
print(f'Wrote submission.csv: {len(submission)} rows '
      f'({(submission.row_type=="node").sum()} nodes, {(submission.row_type=="edge").sum()} edges)')

# ---- sanity checks: catch format bugs before you burn a submission ----
assert list(submission.columns) == ['dataset','row_type','node_id','t','z','y','x','source_id','target_id']
assert set(datasets).issubset(set(submission.dataset.unique())), 'every test dataset must appear'
nodes = submission[submission.row_type == 'node']
edges = submission[submission.row_type == 'edge']
assert (nodes[['t','z','y','x']] >= 0).all().all(), 'node coords must be non-negative'
# edges must reference real node_ids within the same dataset
for ds, grp in submission.groupby('dataset'):
    nset = set(grp.loc[grp.row_type=='node', 'node_id'])
    e = grp[grp.row_type=='edge']
    assert e['source_id'].isin(nset).all() and e['target_id'].isin(nset).all(), f'dangling edge in {ds}'
print('All sanity checks passed ✅')
submission.head()

Wrote submission.csv: 78063 rows (41325 nodes, 36738 edges)
All sanity checks passed ✅


,dataset,row_type,node_id,t,z,y,x,source_id,target_id
id,,,,,,,,,
0,44b6_0113de3b,node,1,0,41,6,174,-1,-1
1,44b6_0113de3b,node,2,0,48,102,170,-1,-1
2,44b6_0113de3b,node,3,0,35,94,130,-1,-1
3,44b6_0113de3b,node,4,0,51,138,174,-1,-1
4,44b6_0113de3b,node,5,0,62,46,230,-1,-1


## 5 · Where to push next (ideas that raise the ceiling)

This is a strong *classical* baseline. Concrete upgrades, roughly by effort:

1. **Calibrate detection on the training set.** Each train `.geff` exposes `estimated_number_of_nodes`. Sweep `MIN_PEAK_DIST` / `THRESH_REL` so your detected count matches it — this directly counters the over-prediction penalty.
2. **Watershed split for dense clusters.** Replace pure peak-picking with marker-controlled watershed (`distance_transform_edt(..., sampling=SCALE)` for physically correct seeds) to recover cells that merge under high density.
3. **Smarter linking cost.** Add a small motion/velocity prior (predict position from the previous displacement), or solve a global multi-frame assignment instead of greedy frame pairs.
4. **Learn detection.** Drop in a pretrained **Cellpose** / **StarDist-3D** model (attach weights as a Kaggle dataset — internet is off but datasets are allowed). This is usually the biggest single jump in detection quality.
5. **Gap closing.** Allow links across `t → t+2` to bridge frames where a cell is briefly missed, which also stabilises lineages and divisions.

Tune the knobs in the **CONFIG** block against the leaderboard, A/B test `DETECT_DIVISIONS`, and iterate. Good luck! 🚀